# Sel 1: Instalasi dan impor

In [1]:
%pip install pandas tqdm

import time
import urllib.request
from ftplib import FTP
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from tqdm.auto import tqdm

Note: you may need to restart the kernel to use updated packages.


/opt/anaconda3/envs/biochem/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Sel 2: Konfigurasi

In [2]:
DATA_DIR = Path("data_bvbrc")
FAA_DIR = DATA_DIR / "faa"
FAA_DIR.mkdir(parents=True, exist_ok=True)

SPECIES_KEYWORD = "Mycobacterium tuberculosis"   # ganti sesuai target
ANTIBIOTIC = "rifampicin"                        # ganti sesuai target (huruf kecil)
MAX_GENOMES = 20                                 # batasi dulu untuk uji coba, isi None untuk semua
N_WORKERS = 4                                    # jangan terlalu besar agar server FTP tidak terbebani

In [3]:
from ftplib import FTP, FTP_TLS

class ReusedSessionFTP_TLS(FTP_TLS):
    """FTP_TLS yang memakai ulang sesi TLS kanal kontrol pada kanal data."""
    def ntransfercmd(self, cmd, rest=None):
        conn, size = FTP.ntransfercmd(self, cmd, rest)
        if self._prot_p:
            conn = self.context.wrap_socket(
                conn, server_hostname=self.host, session=self.sock.session
            )
        return conn, size

def ftps_connect(host="ftp.bvbrc.org", timeout=60):
    ftp = ReusedSessionFTP_TLS()
    ftp.connect(host, 21, timeout=timeout)
    ftp.login()        # anonymous
    ftp.prot_p()       # enkripsi kanal data
    return ftp

def ftps_download(ftp, remote_path, local_path):
    local_path = Path(local_path)
    tmp = local_path.with_suffix(local_path.suffix + ".part")
    with open(tmp, "wb") as f:
        ftp.retrbinary(f"RETR {remote_path}", f.write)
    tmp.replace(local_path)

# Sel 3: Unduh tabel fenotipe AMR

In [ ]:
AMR_REMOTE = "/RELEASE_NOTES/PATRIC_genomes_AMR.txt"
AMR_FILE = DATA_DIR / "PATRIC_genomes_AMR.txt"

if not AMR_FILE.exists():
    ftp = ftps_connect()
    try:
        ftps_download(ftp, AMR_REMOTE, AMR_FILE)
    finally:
        ftp.quit()

amr = pd.read_csv(AMR_FILE, sep="\t", dtype=str, low_memory=False)
amr.columns = amr.columns.str.strip().str.lower()
print(amr.shape)
print(list(amr.columns))
amr.head()

(1207585, 16)
['genome_id', 'genome_name', 'taxon_id', 'antibiotic', 'resistant_phenotype', 'measurement', 'measurement_sign', 'measurement_value', 'measurement_unit', 'laboratory_typing_method', 'laboratory_typing_method_version', 'laboratory_typing_platform', 'vendor', 'testing_standard', 'testing_standard_year', 'source']


,genome_id,genome_name,taxon_id,antibiotic,resistant_phenotype,measurement,measurement_sign,measurement_value,measurement_unit,laboratory_typing_method,laboratory_typing_method_version,laboratory_typing_platform,vendor,testing_standard,testing_standard_year,source
0,562.144637,"""Escherichia coli """,562,amoxicillin/clavulanic acid,NaN,2,NaN,2,mg/L,MIC,NaN,NaN,NaN,CLSI,2018,NaN
1,562.144637,"""Escherichia coli """,562,ampicillin,NaN,<=1,<=,1,mg/L,MIC,NaN,NaN,NaN,CLSI,2018,NaN
2,562.144637,"""Escherichia coli """,562,azithromycin,NaN,2,NaN,2,mg/L,MIC,NaN,NaN,NaN,CLSI,2018,NaN
3,562.144637,"""Escherichia coli """,562,cefoxitin,NaN,2,NaN,2,mg/L,MIC,NaN,NaN,NaN,CLSI,2018,NaN
4,562.144637,"""Escherichia coli """,562,ceftriaxone,NaN,<=0.25,<=,0.25,mg/L,MIC,NaN,NaN,NaN,CLSI,2018,NaN


# Sel 4: Ringkasan jumlah data per spesies dan antibiotik

In [10]:
need = {"genome_id", "genome_name", "antibiotic", "resistant_phenotype"}
missing = need - set(amr.columns)
assert not missing, f"Kolom tidak ditemukan: {missing}. Cek nama kolom pada output Sel 3."

amr["species"] = (amr["genome_name"].str.replace('"', "", regex=False).str.strip()
                  .str.split().str[:2].str.join(" "))
s = amr[amr["resistant_phenotype"].isin(["Susceptible", "Resistant"])]
summ = (s.groupby(["species", "antibiotic", "resistant_phenotype"])["genome_id"]
          .nunique().unstack(fill_value=0))
summ["total"] = summ.sum(axis=1)
summ["minor_pct"] = (summ[["Resistant", "Susceptible"]].min(axis=1) / summ["total"] * 100).round(1)
summ.sort_values("total", ascending=False).head(30)

resistant_phenotype                                       Resistant  \
species                    antibiotic                                 
Mycobacterium tuberculosis rifampin                            7065   
                           isoniazid                           8372   
                           ethambutol                          4718   
                           pyrazinamide                        2119   
Streptococcus pneumoniae   penicillin                          4359   
                           trimethoprim/sulfamethoxazole       6006   
                           erythromycin                        2984   
                           chloramphenicol                      657   
                           tetracycline                        2743   
Mycobacterium tuberculosis streptomycin                        3032   
Salmonella enterica        ampicillin                          2970   
                           chloramphenicol                     1563   
Escherichia coli           gentamicin                           864   
                           ciprofloxacin                       1818   
                           ceftazidime                          997   
Salmonella enterica        ciprofloxacin                       1133   
                           tetracycline                        2486   
Escherichia coli           cefotaxime                          1251   
                           ampicillin                          3645   
                           piperacillin/tazobactam              473   
Salmonella enterica        gentamicin                           551   
                           streptomycin                        1620   
Escherichia coli           meropenem                             82   
Salmonella enterica        azithromycin                         187   
Neisseria gonorrhoeae      ceftriaxone                           41   
                           azithromycin                         747   
Mycobacterium tuberculosis amikacin                             732   
Escherichia coli           cefuroxime                           689   
Mycobacterium tuberculosis kanamycin                            780   
Klebsiella pneumoniae      gentamicin                          1763   

resistant_phenotype                                       Susceptible  total  \
species                    antibiotic                                          
Mycobacterium tuberculosis rifampin                             16115  23180   
                           isoniazid                            14538  22910   
                           ethambutol                           16428  21146   
                           pyrazinamide                         11462  13581   
Streptococcus pneumoniae   penicillin                            8982  13341   
                           trimethoprim/sulfamethoxazole         6464  12470   
                           erythromycin                          8303  11287   
                           chloramphenicol                       8845   9502   
                           tetracycline                          6092   8835   
Mycobacterium tuberculosis streptomycin                          5569   8601   
Salmonella enterica        ampicillin                            5161   8131   
                           chloramphenicol                       6490   8053   
Escherichia coli           gentamicin                            6482   7346   
                           ciprofloxacin                         5471   7289   
                           ceftazidime                           6034   7031   
Salmonella enterica        ciprofloxacin                         5890   7023   
                           tetracycline                          4226   6712   
Escherichia coli           cefotaxime                            5298   6549   
                           ampicillin                            2822   6467   
                           piperacillin/tazobac

# Sel 5: Filter data sesuai target

In [17]:
ANTIBIOTIC_ALIASES = ["rifampin"] 

df = amr.copy()
print("Awal:", len(df))

# 1) Spesies
df = df[df["genome_name"].str.contains(SPECIES_KEYWORD, case=False, regex=False, na=False)]
print("Setelah filter spesies:", len(df))

# 2) Antibiotik
df = df[df["antibiotic"].str.strip().str.lower().isin(ANTIBIOTIC_ALIASES)]
print("Setelah filter antibiotik:", len(df))

# 3) Hanya Susceptible dan Resistant
df["resistant_phenotype"] = df["resistant_phenotype"].str.strip().str.capitalize()
df = df[df["resistant_phenotype"].isin(["Susceptible", "Resistant"])]
print("Setelah filter S/R:", len(df))

assert len(df) > 0, "Hasil filter kosong. Lihat tahap mana yang turun ke 0 di atas."

# Informasi metode uji (hanya ditampilkan, belum dipakai menyaring)
for col in ["laboratory_typing_method", "testing_standard", "source"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False).head(8))

# 4) Buang genom dengan label bertentangan, lalu hapus duplikat
n_label = df.groupby("genome_id")["resistant_phenotype"].nunique()
conflict = n_label[n_label > 1].index
print("\nGenom dengan label bertentangan (dibuang):", len(conflict))
df = df[~df["genome_id"].isin(conflict)]

labels = (df[["genome_id", "genome_name", "antibiotic", "resistant_phenotype"]]
          .drop_duplicates("genome_id").reset_index(drop=True))
labels["label"] = (labels["resistant_phenotype"] == "Resistant").astype(int)

print("\nDistribusi kelas (seluruh data terfilter):")
print(labels["resistant_phenotype"].value_counts())

if MAX_GENOMES:
    labels = labels.sample(n=min(MAX_GENOMES, len(labels)), random_state=42).reset_index(drop=True)
labels.to_csv(DATA_DIR / "labels.csv", index=False)
labels.head()

Awal: 1207585
Setelah filter spesies: 251897
Setelah filter antibiotik: 33880
Setelah filter S/R: 24282

laboratory_typing_method:
laboratory_typing_method
Broth dilution    13428
NaN                6568
MIC                2038
Agar dilution      1996
Disk diffusion      252
Name: count, dtype: int64

testing_standard:
testing_standard
NaN                                                                    20481
WHO: Guidelines for Surveillance of Drug Resistance in Tuberculosis     2429
CLSI                                                                    1372
Name: count, dtype: int64

source:
source
NaN    24282
Name: count, dtype: int64

Genom dengan label bertentangan (dibuang): 1

Distribusi kelas (seluruh data terfilter):
resistant_phenotype
Susceptible    16114
Resistant       7064
Name: count, dtype: int64


,genome_id,genome_name,antibiotic,resistant_phenotype,label
0,1773.1568,Mycobacterium tuberculosis strain 0268S,rifampin,Susceptible,0
1,1733.1247,Mycobacterium tuberculosis 0516M,rifampin,Susceptible,0
2,1733.11166,Mycobacterium tuberculosis 13s405,rifampin,Susceptible,0
3,1773.4741,Mycobacterium tuberculosis C00014898,rifampin,Susceptible,0
4,1733.4794,Mycobacterium tuberculosis 13.0612462,rifampin,Susceptible,0


In [18]:
tmp = df[["genome_id", "resistant_phenotype",
          "laboratory_typing_method", "testing_standard"]].copy()
tmp["laboratory_typing_method"] = tmp["laboratory_typing_method"].fillna("tidak_diketahui")
tmp["testing_standard"] = tmp["testing_standard"].fillna("tidak_diketahui")

for col in ["laboratory_typing_method", "testing_standard"]:
    print(f"\n{col} (jumlah baris):")
    print(pd.crosstab(tmp[col], tmp["resistant_phenotype"], margins=True))
    print(f"\n{col} (persentase per baris):")
    print((pd.crosstab(tmp[col], tmp["resistant_phenotype"], normalize="index") * 100).round(1))


laboratory_typing_method (jumlah baris):
resistant_phenotype       Resistant  Susceptible    All
laboratory_typing_method                               
Agar dilution                  1240          755   1995
Broth dilution                 3703         9724  13427
Disk diffusion                  192           60    252
MIC                            1087          951   2038
tidak_diketahui                1384         5184   6568
All                            7606        16674  24280

laboratory_typing_method (persentase per baris):
resistant_phenotype       Resistant  Susceptible
laboratory_typing_method                        
Agar dilution                  62.2         37.8
Broth dilution                 27.6         72.4
Disk diffusion                 76.2         23.8
MIC                            53.3         46.7
tidak_diketahui                21.1         78.9

testing_standard (jumlah baris):
resistant_phenotype                                 Resistant  Susceptible  \
testi

In [19]:
g = (df[["genome_id", "resistant_phenotype", "laboratory_typing_method"]]
     .assign(laboratory_typing_method=lambda d: d["laboratory_typing_method"].fillna("tidak_diketahui")))

# Berapa genom yang punya lebih dari satu metode?
n_method = g.groupby("genome_id")["laboratory_typing_method"].nunique()
print("Genom dengan >1 metode:", (n_method > 1).sum(), "dari", len(n_method))

# Ukuran subset per metode, di tingkat genom (genom bermetode ganda dihitung di tiap metode)
sub = g.drop_duplicates(["genome_id", "laboratory_typing_method"])
print(pd.crosstab(sub["laboratory_typing_method"], sub["resistant_phenotype"], margins=True))

Genom dengan >1 metode: 79 dari 23178
resistant_phenotype       Resistant  Susceptible    All
laboratory_typing_method                               
Agar dilution                  1240          755   1995
Broth dilution                 3207         9197  12404
Disk diffusion                  192           60    252
MIC                            1087          951   2038
tidak_diketahui                1384         5184   6568
All                            7110        16147  23257


# Sel 6: Verifikasi lokasi file protein di FTP

In [20]:
gid = labels["genome_id"].iloc[0]
ftp = ftps_connect()
try:
    ftp.cwd(f"/genomes/{gid}")
    print(ftp.nlst())
finally:
    ftp.quit()

['1773.1568.PATRIC.spgene.tab', '1773.1568.PATRIC.pathway.tab', '1773.1568.PATRIC.frn', '1773.1568.PATRIC.gff', '1773.1568.PATRIC.features.tab', '1773.1568.PATRIC.faa', '1773.1568.PATRIC.ffn', '1773.1568.fna', '1773.1568.PATRIC.subsystem.tab']


# Sel 7: Unduh file protein (.faa)

In [21]:
FAA_REMOTE_TEMPLATE = "/genomes/{gid}/{gid}.PATRIC.faa"

def worker(gids):
    ftp, ok_list, fail_list = None, [], []
    for gid in gids:
        out = FAA_DIR / f"{gid}.faa"
        if out.exists() and out.stat().st_size > 0:
            ok_list.append(gid)
            continue
        success = False
        for _ in range(3):
            try:
                if ftp is None:
                    ftp = ftps_connect()
                ftps_download(ftp, FAA_REMOTE_TEMPLATE.format(gid=gid), out)
                success = True
                break
            except Exception:
                try:
                    if ftp is not None:
                        ftp.close()
                except Exception:
                    pass
                ftp = None
                time.sleep(2)
        (ok_list if success else fail_list).append(gid)
    try:
        if ftp is not None:
            ftp.quit()
    except Exception:
        pass
    return ok_list, fail_list

gids = labels["genome_id"].tolist()
chunks = [gids[i::N_WORKERS] for i in range(N_WORKERS)]

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    results = list(ex.map(worker, chunks))

ok = [g for r in results for g in r[0]]
failed = [g for r in results for g in r[1]]
print("Berhasil:", len(ok), "| Gagal:", len(failed))
pd.Series(failed).to_csv(DATA_DIR / "failed_genomes.txt", index=False, header=False)

Berhasil: 20 | Gagal: 0


# Sel 8: Baca FASTA dan gabungkan dengan label

In [22]:
def read_fasta(path):
    header, seq = None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq)
                header, seq = line[1:], []
            else:
                seq.append(line)
    if header is not None:
        yield header, "".join(seq)

rows = []
for gid in labels["genome_id"]:
    path = FAA_DIR / f"{gid}.faa"
    if not path.exists():
        continue
    for header, seq in read_fasta(path):
        rows.append((gid, header, seq))

proteins = pd.DataFrame(rows, columns=["genome_id", "header", "sequence"])
data = proteins.merge(labels[["genome_id", "resistant_phenotype", "label"]],
                      on="genome_id", how="inner")

print(data.shape)
print(data["genome_id"].nunique(), "genom")
data.to_csv(DATA_DIR / "proteins_labeled.csv.gz", index=False, compression="gzip")
data.head()

(88952, 5)
20 genom


,genome_id,header,sequence,resistant_phenotype,label
0,1773.1568,fig|1773.1568.peg.1 PE_PGRS family protein =...,MSFLIASPEALAATATYLTGIGSAISAANAVAAAPTTEILAAGTDE...,Susceptible,0
1,1773.1568,fig|1773.1568.peg.2 Acid and phagosome regul...,MAKVTAARPEPSAAWAEARRRVRQRREDMLRHPAFLSKQLPAEPAD...,Susceptible,0
2,1773.1568,fig|1773.1568.peg.3|ERS094118_00003 Oligopep...,MSGATVGAREITIRGVVLGALITLVFTAANVYLGLRVGLTFATSIP...,Susceptible,0
3,1773.1568,fig|1773.1568.peg.4|ERS094118_00004 Gamma-gl...,MLSLSGCGGFHAGAPSTAGPCEIVPNGTPAPKTPPATVPSSRNLAT...,Susceptible,0
4,1773.1568,fig|1773.1568.peg.5|ERS094118_00005 Sirohydr...,MTAPATMQSAAMLRSGAIEAPPATMQSAAMRWGHLPLAEESGTIAP...,Susceptible,0
